In [1]:
import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')  
gpus = tf.config.list_physical_devices('GPU')

import pandas as pd
from utils import encode_labels, get_class_weights, plot_confusion_matrix

data = "./features-relative.csv"
df = pd.read_csv(data)
patient_id = "BA0803901"

df_test = df[df['file'].str.contains(patient_id)]

# Train set: all other patients
df_train = df[~df['file'].str.contains(patient_id)]

X = df.drop(['Label', 'Start', 'End', 'file', 'Start Sample', 'End Sample'], axis=1)
X_train = df_train.drop(['Label', 'Start', 'End', 'file', 'Start Sample', 'End Sample'], axis=1)
X_test = df_test.drop(['Label', 'Start', 'End', 'file', 'Start Sample', 'End Sample'], axis=1)

# Labels
y_train = df_train['Label']
y_test = df_test['Label']

y_train, y_text = encode_labels(y_train, y_test)


2025-12-11 14:51:03.260589: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-11 14:51:03.333097: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Load data
data = "./features-relative.csv"
df = pd.read_csv(data)

# Feature matrix X (drop non-feature columns)
X = df.drop(['Label', 'Start', 'End', 'file', 'Start Sample', 'End Sample'], axis=1)

# Labels
y = df['Label']

le = LabelEncoder()
y = le.fit_transform(y)

# Standard train/test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

y_train, y_test = encode_labels(y_train, y_test)


In [3]:

from sklearn.ensemble import RandomForestClassifier


class_weights = get_class_weights(y_train)
rfc = RandomForestClassifier(random_state=0, n_estimators=300, class_weight=get_class_weights(y_train))

rfc.fit(X_train, y_train)

,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [ ]:
y_pred = rfc.predict(X_test)

# class_report = classification_report(y_test, y_pred)
# print(class_report)
plot_confusion_matrix(y_test,y_pred, ["Fibrillation", "PSW", "None"])

TypeError: plot_confusion_matrix() missing 1 required positional argument: 'labels'

In [ ]:
feature_scores = pd.Series(rfc.feature_importances_, index=X_train.columns).sort_values(ascending=False)

print(feature_scores)

spec_entropy_3    0.027436
spec_entropy_6    0.025333
spec_entropy_5    0.024595
spec_entropy_4    0.023864
spec_entropy_2    0.023752
                    ...   
zcr_8             0.000363
zcr_2             0.000359
zcr_5             0.000357
zcr_7             0.000356
zcr_9             0.000314
Length: 120, dtype: float64


Accuracy with normal split 0.89 accuracy
with patient split 0.78